# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides an interactive guide for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the `mlcroissant` library, following Croissant schema best practices. All references to data elements are made via their `@id` fields for reproducibility and clarity.

### Dataset Source
Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install -q mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata via Croissant
dataset = mlc.Dataset(croissant_url)
# Metadata is an object, not a dict: access properties as attributes
name = dataset.metadata.name
description = dataset.metadata.description
print(f"{name}: {description}")

## 2. Data Overview

Explore available record sets and their fields. All entities are referenced by their `@id`. This is a crucial step to understand how the Croissant package organizes data for programmatic access.

Below, we print the list of record set `@id`s and enumerate the available fields for each.

In [ ]:
# List record set IDs and their fields
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}")

for record_set in record_sets:
    print(f"\nRecord set: {record_set['@id']}")
    # List all fields in this record set
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            print(f"  Field: {field['@id']} (name: {field.get('name', field.get('@id',''))})")
        elif isinstance(field, str):
            print(f"  Field: {field}")

Often you'll want to preview a sample of the records for a particular record set. Replace `<record_set_id>` below with the `@id` of the record set you'd like to examine. For demonstration, we'll use the **first** record set found above.

In [ ]:
# Show a sample of the records for the first record set
if record_sets:
    first_record_set_id = record_sets[0]['@id']
    print(f"\nSample records for record set: {first_record_set_id}")
    for idx, rec in enumerate(dataset.records(record_set=first_record_set_id)):
        if idx >= 2:
            break
        print(json.dumps(rec, indent=2))

## 3. Data Extraction

Extract full data from selected record sets into Pandas DataFrames for analysis. All access uses `@id` fields.

In [ ]:
# Gather all record set IDs
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"Available record sets: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    # Extract all records
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"\nLoaded DataFrame for: {record_set_id}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head())
    else:
        print(f"No records found for record set: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Processing and summarizing typical numeric fields. We demonstrate filtering, normalization, and grouping by a categorical field. All variable names use their `@id` for traceability.

**Note:** Replace `<numeric_field_id>` and `<group_field_id>` with actual field `@id`s from your data above. For demonstration, we find a numeric field automatically if possible.

In [ ]:
import numpy as np

# Select a record set to analyze (use the first one with data)
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        break

if main_record_set_id is None:
    raise ValueError('No record set with data found.')

df = dataframes[main_record_set_id]
# Find a numeric field by inspecting dtypes or column names
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
    # Try to coerce columns to numeric if possible
    try:
        coerced = pd.to_numeric(df[col])
        if not coerced.isnull().all():
            df[col] = coerced
            numeric_field_id = col
            break
    except Exception:
        continue

if numeric_field_id is None:
    raise ValueError('No numeric field detected.')

print(f"Using numeric field: {numeric_field_id}")

threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile for filtering
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to find a group field (categorical variable with a reasonable number of unique values)
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and df[col].nunique() > 1 and df[col].nunique() < min(10, len(df)//2):
        group_field_id = col
        break

if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df)
else:
    print('No suitable group field found for grouping analysis.')

## 5. Visualization

We visualize the distribution of the selected numeric field, and if available, compare distributions by the group field using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If group_field_id exists, plot by group
if group_field_id is not None and group_field_id in df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we loaded, inspected, and explored the dataset using Croissant's `@id`-based access and the `mlcroissant` Python API. We demonstrated how to select fields by their canonical identifiers, perform filtering and grouping EDA, and visualize key field distributions for model-ready data understanding.

Key steps you followed:
- Loaded Croissant metadata from schema URL, retrieved dataset and record set structure by `@id`.
- Discovered available fields per record set and loaded records into Pandas DataFrames.
- Identified numeric and categorical (grouping) fields by inspecting `@id`s and sample values.
- Applied standard filtering, normalization, and grouping using only `@id` references for full reproducibility.
- Plotted distributions and summaries to guide further quantitative analysis and reporting.

*Next steps*: Consult the Croissant schema for specific `@id` documentation and field semantics for advanced analysis. Happy exploring!